[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/study_notes/week08_sequential_testing.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 8: Sequential Testing (Wald's SPRT)

**Course**: Advanced A/B Testing | **Context**: QuickCart (online electronics retailer)

**Your role**: Data scientist on the experimentation platform team. QuickCart runs dozens of experiments simultaneously, and product managers want to ship winning variants as quickly as possible. But peeking at results early inflates false positives. This week you'll learn **Sequential Probability Ratio Test (SPRT)** -- a principled framework that lets you monitor experiments continuously and stop early *without* inflating your error rates.

---

## What You'll Learn

1. The peeking problem: why checking results early inflates false positives
2. Sequential testing as the principled alternative to peeking
3. Wald's Sequential Probability Ratio Test (SPRT) -- intuition and math
4. Log-likelihood ratio accumulation for normal data
5. Decision boundaries: upper (reject H0), lower (reject H1), continue zone
6. Implementing SPRT from scratch
7. Simulation: SPRT vs fixed-horizon test (sample savings)
8. Batch SPRT (processing data in daily batches rather than one-at-a-time)
9. Operating characteristics: expected sample size under H0 and H1
10. Practical considerations: choosing effect_size, when to use sequential vs fixed

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

from collections import defaultdict

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---

## 1. The Peeking Problem

### Why This Matters at QuickCart

QuickCart's PM for the checkout team just launched an experiment on Tuesday. By Thursday, she checks the dashboard: p=0.03! She messages: "The new checkout flow is significant -- can we ship it?"

The problem: **repeated testing inflates the false positive rate**. A test designed for 5% Type I error at the end of the planned duration can produce false positives at 20-30% rates when checked multiple times.

### Intuition

Under the null hypothesis (no real effect), the test statistic follows a random walk. If you check repeatedly, you're essentially asking: "Did this random walk *ever* cross the threshold?" -- which is much more likely than asking "Is it beyond the threshold at exactly time $T$?"

Let's demonstrate this empirically.

In [ ]:
def simulate_peeking(n_total, n_peeks, n_simulations=10000, alpha=0.05):
    """
    Simulate the false positive rate when peeking at results multiple times.
    
    Both groups come from the same distribution (null is true).
    We check for significance at n_peeks equally-spaced points.
    """
    false_positives = 0
    peek_points = np.linspace(n_total // n_peeks, n_total, n_peeks).astype(int)
    
    for _ in range(n_simulations):
        # Generate data under null: both from N(0, 1)
        control = np.random.normal(0, 1, n_total)
        treatment = np.random.normal(0, 1, n_total)  # same distribution!
        
        for n in peek_points:
            _, p_value = stats.ttest_ind(treatment[:n], control[:n])
            if p_value < alpha:
                false_positives += 1
                break  # stop at first significant result (ship it!)
    
    return false_positives / n_simulations


# How does the number of peeks affect the false positive rate?
peek_counts = [1, 2, 5, 10, 20, 50, 100]
fpr_values = []

print("False Positive Rate vs Number of Peeks (null is true everywhere):")
print(f"{'Peeks':>6} {'FPR':>8}")
print("-" * 16)

for n_peeks in peek_counts:
    fpr = simulate_peeking(n_total=1000, n_peeks=n_peeks, n_simulations=10000)
    fpr_values.append(fpr)
    print(f"{n_peeks:>6} {fpr:>8.3f}")

plt.plot(peek_counts, fpr_values, 'bo-', linewidth=2, markersize=8)
plt.axhline(0.05, color='red', linestyle='--', linewidth=2, label='Nominal alpha = 0.05')
plt.xlabel('Number of peeks during experiment')
plt.ylabel('Actual False Positive Rate')
plt.title('The Peeking Problem: More Checks = More False Positives')
plt.xscale('log')
plt.legend()
plt.grid(alpha=0.3)
plt.ylim([0, 0.35])
plt.show()

print(f"\nWith 100 peeks, the actual FPR is ~{fpr_values[-1]:.0%} -- "
      f"that's {fpr_values[-1]/0.05:.1f}x the intended rate!")

### Visualizing the Random Walk Under the Null

Let's trace the p-value path over time for a single experiment where there is NO real effect. Notice how it dips below 0.05 at various points -- each dip would be a false "ship it!" moment if you stopped there.

In [ ]:
np.random.seed(7)  # chosen to show an interesting path
n_total = 2000

control = np.random.normal(0, 1, n_total)
treatment = np.random.normal(0, 1, n_total)  # NO effect

sample_points = np.arange(50, n_total + 1, 10)
p_values = []

for n in sample_points:
    _, p = stats.ttest_ind(treatment[:n], control[:n])
    p_values.append(p)

plt.semilogy(sample_points, p_values, 'b-', alpha=0.7, linewidth=1.5)
plt.axhline(0.05, color='red', linestyle='--', linewidth=2, label='alpha = 0.05')
plt.fill_between(sample_points, 0, 0.05, alpha=0.1, color='red', label='"Significant" zone')
plt.xlabel('Sample size per group')
plt.ylabel('p-value (log scale)')
plt.title('P-value Path Over Time (NULL is true -- no real effect)')
plt.legend()
plt.grid(alpha=0.3)
plt.ylim([0.001, 1])
plt.show()

n_dips = sum(1 for p in p_values if p < 0.05)
print(f"The p-value dipped below 0.05 at {n_dips} check points out of {len(sample_points)}")
print(f"If you stopped at the first dip, you'd ship a null effect as 'significant'.")

---

## 2. Sequential Testing: The Principled Alternative

### The Key Insight

The peeking problem arises because fixed-horizon tests were designed for a **single look** at the data. If you want to look continuously, you need a test **designed for continuous monitoring**.

**Sequential testing** frameworks provide exactly this:
- You can check after every observation (or batch)
- Error rates ($\alpha$, $\beta$) are controlled across ALL possible stopping times
- When there IS a real effect, you stop early and save samples
- When there is NOT an effect, you also stop early (accepting the null)

### Wald's SPRT (1945)

The **Sequential Probability Ratio Test** was developed by Abraham Wald during WWII for quality control in manufacturing. The core idea:

1. After each observation, compute the **likelihood ratio** -- how much more likely is the data under H1 vs H0?
2. Accumulate evidence as a running sum of log-likelihood ratios
3. Compare against two boundaries:
   - **Upper boundary** (B): enough evidence to reject H0 (effect is real)
   - **Lower boundary** (A): enough evidence to reject H1 (no meaningful effect)
   - **Between A and B**: inconclusive, keep collecting data

### Why This Works

The boundaries are set so that:
- P(cross upper boundary | H0 true) $\leq \alpha$ (Type I error control)
- P(cross lower boundary | H1 true) $\leq \beta$ (Type II error control)

No matter WHEN you stop, these guarantees hold.

---

## 3. SPRT: The Mathematics

### Setting

We test:
- $H_0$: The treatment effect $\mu = 0$ (no difference)
- $H_1$: The treatment effect $\mu = \delta$ (a specific alternative)

where $\delta$ is the **minimum detectable effect** we care about.

### The Likelihood Ratio

For observation $x_i$ from a normal distribution with known variance $\sigma^2$:

$$
\Lambda_n = \prod_{i=1}^n \frac{f(x_i | \mu = \delta)}{f(x_i | \mu = 0)} = \prod_{i=1}^n \frac{\exp\left(-\frac{(x_i - \delta)^2}{2\sigma^2}\right)}{\exp\left(-\frac{x_i^2}{2\sigma^2}\right)}
$$

Taking the logarithm (the **log-likelihood ratio**):

$$
\log \Lambda_n = \sum_{i=1}^n \left[ \frac{\delta}{\sigma^2} \cdot x_i - \frac{\delta^2}{2\sigma^2} \right] = \frac{\delta}{\sigma^2} \sum_{i=1}^n x_i - \frac{n\delta^2}{2\sigma^2}
$$

Or equivalently, for each observation the individual contribution is:

$$
\log \frac{f_1(x_i)}{f_0(x_i)} = \frac{\delta}{\sigma^2} \left(x_i - \frac{\delta}{2}\right)
$$

### Decision Boundaries

- **Upper boundary** $B = \log\frac{1-\beta}{\alpha}$: reject $H_0$ (conclude effect is real)
- **Lower boundary** $A = \log\frac{\beta}{1-\alpha}$: reject $H_1$ (conclude no meaningful effect)

### Decision Rule

$$
\text{Decision} = \begin{cases}
\text{Reject } H_0 & \text{if } \log \Lambda_n \geq B \\
\text{Reject } H_1 & \text{if } \log \Lambda_n \leq A \\
\text{Continue} & \text{if } A < \log \Lambda_n < B
\end{cases}
$$

:::{admonition} Full SPRT derivation and boundary justification (click to expand)
:class: dropdown

### Why These Boundaries Control Error Rates

Let $\alpha$ = P(reject $H_0$ | $H_0$ true) and $\beta$ = P(reject $H_1$ | $H_1$ true).

**Upper boundary derivation:**

When $H_0$ is true, the probability of ever crossing the upper boundary $B$ satisfies:

$$
P(\Lambda_n \geq e^B | H_0) \leq \alpha
$$

Wald showed that setting $B = \log\frac{1-\beta}{\alpha}$ approximately satisfies this constraint. The approximation arises from "overshooting" -- the test statistic may jump past the boundary rather than landing exactly on it.

**Lower boundary derivation:**

When $H_1$ is true, the probability of ever crossing the lower boundary $A$ satisfies:

$$
P(\Lambda_n \leq e^A | H_1) \leq \beta
$$

Setting $A = \log\frac{\beta}{1-\alpha}$ approximately satisfies this.

**The Wald equations:**

From the error probability constraints, Wald derives the following system. Let $\alpha'$ be the actual Type I rate and $\beta'$ the actual Type II rate:

$$
\frac{\alpha'}{1 - \beta'} \leq e^{-B} = \frac{\alpha}{1-\beta}
$$

$$
\frac{1 - \alpha'}{\beta'} \geq e^{-A} = \frac{1 - \alpha}{\beta}
$$

These inequalities show that the actual error rates are bounded by the nominal levels, making SPRT slightly conservative in practice.

**Numerical example:** With $\alpha = 0.05$, $\beta = 0.2$:
- $B = \log(0.8 / 0.05) = \log(16) \approx 2.77$
- $A = \log(0.2 / 0.95) = \log(0.211) \approx -1.56$

**Log-likelihood ratio for normal observations:**

Given $x_i \sim N(\mu, \sigma^2)$ under either hypothesis, with $\mu_0 = 0$ and $\mu_1 = \delta$:

$$
\log \frac{f(x_i | \mu_1)}{f(x_i | \mu_0)} = \frac{(\mu_1 - \mu_0)}{\sigma^2}\left(x_i - \frac{\mu_0 + \mu_1}{2}\right) = \frac{\delta}{\sigma^2}\left(x_i - \frac{\delta}{2}\right)
$$

The cumulative sum $\sum_{i=1}^n \frac{\delta}{\sigma^2}(x_i - \delta/2)$ is the test statistic compared against boundaries $A$ and $B$.

**Expected drift:**
- Under $H_0$ ($x_i \sim N(0, \sigma^2)$): $E[\text{increment}] = \frac{\delta}{\sigma^2}(0 - \delta/2) = -\frac{\delta^2}{2\sigma^2} < 0$ (drifts toward A)
- Under $H_1$ ($x_i \sim N(\delta, \sigma^2)$): $E[\text{increment}] = \frac{\delta}{\sigma^2}(\delta - \delta/2) = +\frac{\delta^2}{2\sigma^2} > 0$ (drifts toward B)
:::

---

## 4. Implementing SPRT From Scratch

Let's build a `SequentialTester` class that accumulates evidence and makes decisions.

In [ ]:
class SequentialTester:
    """
    Wald's Sequential Probability Ratio Test (SPRT) for normal data.
    
    Tests H0: mu = 0 vs H1: mu = effect_size
    using observations assumed to come from N(mu, sigma^2).
    
    Parameters
    ----------
    effect_size : float
        The minimum effect size we want to detect (mu under H1).
    alpha : float
        Type I error rate (probability of rejecting H0 when it's true).
    beta : float
        Type II error rate (probability of rejecting H1 when it's true).
    sigma : float
        Known (or estimated) standard deviation of observations.
    """
    
    def __init__(self, effect_size, alpha=0.05, beta=0.2, sigma=1.0):
        self.effect_size = effect_size
        self.alpha = alpha
        self.beta = beta
        self.sigma = sigma
        
        # Decision boundaries (Wald's approximation)
        self.upper_bound = np.log((1 - beta) / alpha)  # B: reject H0
        self.lower_bound = np.log(beta / (1 - alpha))  # A: reject H1
        
        # State
        self.cumulative_sum = 0.0
        self.n_observations = 0
        self.decision = None
        self.history = []  # track the path for visualization
    
    def _log_likelihood_ratio(self, x):
        """
        Compute log-likelihood ratio for a single observation.
        
        log(f1(x)/f0(x)) = (delta/sigma^2) * (x - delta/2)
        where f0 = N(0, sigma^2) and f1 = N(delta, sigma^2)
        """
        delta = self.effect_size
        sigma2 = self.sigma ** 2
        return (delta / sigma2) * (x - delta / 2)
    
    def add_observation(self, x):
        """Process a single observation and check boundaries."""
        if self.decision is not None:
            return self.decision
        
        llr = self._log_likelihood_ratio(x)
        self.cumulative_sum += llr
        self.n_observations += 1
        self.history.append(self.cumulative_sum)
        
        # Check boundaries
        if self.cumulative_sum >= self.upper_bound:
            self.decision = 'reject_H0'  # effect is real
        elif self.cumulative_sum <= self.lower_bound:
            self.decision = 'reject_H1'  # no meaningful effect
        
        return self.decision
    
    def add_batch(self, x_batch):
        """
        Process a batch of observations.
        Returns decision as soon as a boundary is crossed.
        """
        for x in x_batch:
            result = self.add_observation(x)
            if result is not None:
                return result
        return None  # still in continue zone
    
    def run_test(self, data_stream, batch_size=1):
        """
        Process an entire data stream in batches.
        Returns (decision, n_observations_used).
        """
        for i in range(0, len(data_stream), batch_size):
            batch = data_stream[i:i + batch_size]
            result = self.add_batch(batch)
            if result is not None:
                return result, self.n_observations
        return None, self.n_observations  # exhausted data without decision
    
    def reset(self):
        """Reset the tester for a new experiment."""
        self.cumulative_sum = 0.0
        self.n_observations = 0
        self.decision = None
        self.history = []


# Verify the boundaries
tester = SequentialTester(effect_size=0.3, alpha=0.05, beta=0.2)
print(f"SPRT Boundaries (alpha={tester.alpha}, beta={tester.beta}):")
print(f"  Upper boundary (B): {tester.upper_bound:.4f}  (reject H0 = declare effect real)")
print(f"  Lower boundary (A): {tester.lower_bound:.4f}  (reject H1 = declare no effect)")
print(f"  Effect size (delta): {tester.effect_size}")
print(f"  Sigma: {tester.sigma}")
print(f"\nThe cumulative log-likelihood ratio starts at 0 and drifts between A and B.")
print(f"Under H0, expected drift per obs: {-0.3**2/(2*1.0**2):.4f} (toward A)")
print(f"Under H1, expected drift per obs: {+0.3**2/(2*1.0**2):.4f} (toward B)")

---

## 5. Visualizing the Sequential Path

The most intuitive way to understand SPRT is to watch the cumulative log-likelihood ratio walk between the two boundaries. Let's simulate experiments under both hypotheses.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

effect_size = 0.3
sigma = 1.0

# Left panel: Under H0 (no effect)
np.random.seed(42)
ax = axes[0]
B = np.log(0.8 / 0.05)
A = np.log(0.2 / 0.95)
ax.axhline(B, color='green', linestyle='--', linewidth=2, label=f'Upper B = {B:.2f} (reject H0)')
ax.axhline(A, color='red', linestyle='--', linewidth=2, label=f'Lower A = {A:.2f} (reject H1)')
ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
ax.fill_between([0, 500], A, B, alpha=0.05, color='yellow')

for trial in range(8):
    tester = SequentialTester(effect_size=effect_size, sigma=sigma)
    data = np.random.normal(0, sigma, 500)  # H0 is true
    tester.run_test(data)
    color = 'red' if tester.decision == 'reject_H0' else 'blue'
    ax.plot(range(1, len(tester.history) + 1), tester.history, alpha=0.6, color=color, linewidth=1)

ax.set_xlabel('Number of observations')
ax.set_ylabel('Cumulative log-likelihood ratio')
ax.set_title('SPRT paths under H0 (no effect)\nBlue = correct, Red = false positive')
ax.legend(loc='upper left', fontsize=9)
ax.set_ylim([-3, 4])
ax.grid(alpha=0.3)

# Right panel: Under H1 (real effect)
np.random.seed(42)
ax = axes[1]
ax.axhline(B, color='green', linestyle='--', linewidth=2, label=f'Upper B = {B:.2f} (reject H0)')
ax.axhline(A, color='red', linestyle='--', linewidth=2, label=f'Lower A = {A:.2f} (reject H1)')
ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
ax.fill_between([0, 500], A, B, alpha=0.05, color='yellow')

for trial in range(8):
    tester = SequentialTester(effect_size=effect_size, sigma=sigma)
    data = np.random.normal(effect_size, sigma, 500)  # H1 is true
    tester.run_test(data)
    color = 'green' if tester.decision == 'reject_H0' else 'orange'
    ax.plot(range(1, len(tester.history) + 1), tester.history, alpha=0.6, color=color, linewidth=1)

ax.set_xlabel('Number of observations')
ax.set_ylabel('Cumulative log-likelihood ratio')
ax.set_title('SPRT paths under H1 (effect = 0.3)\nGreen = correct, Orange = false negative')
ax.legend(loc='upper left', fontsize=9)
ax.set_ylim([-3, 4])
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Key observations:**

- Under H0 (left): the paths drift downward (toward the lower boundary) on average, correctly concluding "no effect"
- Under H1 (right): the paths drift upward (toward the upper boundary) on average, correctly detecting the effect
- The paths stop as soon as they hit a boundary -- no unnecessary data collection
- Some paths are shorter than others -- the clearer the signal, the faster the decision

---

## 6. Verifying Error Rate Control

Let's run many simulations to confirm that SPRT actually controls the Type I and Type II error rates at the specified levels.

In [ ]:
def verify_error_rates(effect_size, alpha, beta, sigma, n_simulations=10000, max_obs=5000):
    """
    Verify Type I and Type II error rates of SPRT empirically.
    """
    # Under H0
    false_positives = 0
    samples_h0 = []
    
    for _ in range(n_simulations):
        tester = SequentialTester(effect_size=effect_size, alpha=alpha, beta=beta, sigma=sigma)
        data = np.random.normal(0, sigma, max_obs)  # H0 true
        decision, n_obs = tester.run_test(data)
        if decision == 'reject_H0':
            false_positives += 1
        samples_h0.append(n_obs)
    
    # Under H1
    false_negatives = 0
    samples_h1 = []
    
    for _ in range(n_simulations):
        tester = SequentialTester(effect_size=effect_size, alpha=alpha, beta=beta, sigma=sigma)
        data = np.random.normal(effect_size, sigma, max_obs)  # H1 true
        decision, n_obs = tester.run_test(data)
        if decision == 'reject_H1':
            false_negatives += 1
        samples_h1.append(n_obs)
    
    return {
        'type_i_rate': false_positives / n_simulations,
        'type_ii_rate': false_negatives / n_simulations,
        'power': 1 - false_negatives / n_simulations,
        'avg_samples_h0': np.mean(samples_h0),
        'avg_samples_h1': np.mean(samples_h1),
        'median_samples_h0': np.median(samples_h0),
        'median_samples_h1': np.median(samples_h1),
    }


np.random.seed(42)
results = verify_error_rates(effect_size=0.3, alpha=0.05, beta=0.2, sigma=1.0)

print("SPRT Error Rate Verification (effect_size=0.3, alpha=0.05, beta=0.20)")
print("=" * 65)
print(f"\n  Type I error rate:   {results['type_i_rate']:.4f}  (target: <= 0.05)")
print(f"  Type II error rate:  {results['type_ii_rate']:.4f}  (target: <= 0.20)")
print(f"  Power:               {results['power']:.4f}  (target: >= 0.80)")
print(f"\n  Avg samples under H0:    {results['avg_samples_h0']:.0f}")
print(f"  Avg samples under H1:    {results['avg_samples_h1']:.0f}")
print(f"  Median samples under H0: {results['median_samples_h0']:.0f}")
print(f"  Median samples under H1: {results['median_samples_h1']:.0f}")
print(f"\nBoth error rates are at or below their targets -- SPRT is correctly calibrated.")
print(f"(Slightly below target is expected due to boundary overshoot.)")

---

## 7. SPRT vs Fixed-Horizon Test: Sample Savings

### QuickCart Scenario

QuickCart is testing a new product recommendation algorithm. The metric is average order value difference between treatment and control. How much can SPRT save compared to running the full fixed-horizon test?

### Fixed-Horizon Sample Size

For a one-sample z-test with effect size $\delta$, standard deviation $\sigma$, significance level $\alpha$, and power $1-\beta$:

$$
n_{\text{fixed}} = \frac{(z_{\alpha/2} + z_\beta)^2 \sigma^2}{\delta^2}
$$

In [ ]:
def fixed_horizon_sample_size(effect_size, sigma, alpha=0.05, beta=0.2):
    """Compute sample size needed for fixed-horizon z-test (one-sample)."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(1 - beta)
    n = ((z_alpha + z_beta) ** 2 * sigma ** 2) / effect_size ** 2
    return int(np.ceil(n))


def compare_sprt_vs_fixed(effect_size, sigma=1.0, alpha=0.05, beta=0.2,
                           n_simulations=5000, true_effect=None):
    """
    Compare SPRT sample usage vs fixed-horizon test.
    """
    if true_effect is None:
        true_effect = effect_size
    
    n_fixed = fixed_horizon_sample_size(effect_size, sigma, alpha, beta)
    
    sprt_samples = []
    sprt_decisions = []
    
    for _ in range(n_simulations):
        tester = SequentialTester(effect_size=effect_size, alpha=alpha, beta=beta, sigma=sigma)
        data = np.random.normal(true_effect, sigma, n_fixed * 3)  # generous budget
        decision, n_obs = tester.run_test(data)
        sprt_samples.append(n_obs)
        sprt_decisions.append(decision)
    
    return {
        'n_fixed': n_fixed,
        'sprt_mean': np.mean(sprt_samples),
        'sprt_median': np.median(sprt_samples),
        'savings_mean': (1 - np.mean(sprt_samples) / n_fixed) * 100,
        'savings_median': (1 - np.median(sprt_samples) / n_fixed) * 100,
        'sprt_samples': sprt_samples,
    }


np.random.seed(42)

# Three scenarios
result_h1 = compare_sprt_vs_fixed(effect_size=0.3, true_effect=0.3)
result_h0 = compare_sprt_vs_fixed(effect_size=0.3, true_effect=0.0)
result_large = compare_sprt_vs_fixed(effect_size=0.3, true_effect=0.5)

print("SPRT vs Fixed-Horizon Sample Efficiency")
print("=" * 65)
print(f"\nDesign: effect_size=0.3, sigma=1.0, alpha=0.05, beta=0.20")
print(f"Fixed-horizon requires: {result_h1['n_fixed']} observations")
print(f"\n{'Scenario':<30} {'SPRT Mean':>10} {'SPRT Median':>12} {'Savings':>10}")
print("-" * 65)
print(f"{'H1 true (effect=0.3)':<30} {result_h1['sprt_mean']:>10.0f} {result_h1['sprt_median']:>12.0f} {result_h1['savings_mean']:>9.0f}%")
print(f"{'H0 true (no effect)':<30} {result_h0['sprt_mean']:>10.0f} {result_h0['sprt_median']:>12.0f} {result_h0['savings_mean']:>9.0f}%")
print(f"{'H1 true (effect=0.5, larger)':<30} {result_large['sprt_mean']:>10.0f} {result_large['sprt_median']:>12.0f} {result_large['savings_mean']:>9.0f}%")
print(f"\nSPRT saves samples in ALL scenarios -- larger effects are detected faster.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

scenarios = [
    (result_h0, 'H0 true (no effect)', 'steelblue'),
    (result_h1, 'H1 true (effect=0.3)', 'darkorange'),
    (result_large, 'H1 true (effect=0.5)', 'green'),
]

for ax, (result, title, color) in zip(axes, scenarios):
    ax.hist(result['sprt_samples'], bins=50, density=True, alpha=0.7, color=color)
    ax.axvline(result['n_fixed'], color='red', linestyle='--', linewidth=2,
               label=f'Fixed n={result["n_fixed"]}')
    ax.axvline(result['sprt_median'], color='black', linestyle='-', linewidth=2,
               label=f'SPRT median={result["sprt_median"]:.0f}')
    ax.set_xlabel('Samples used')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("When the effect is large, SPRT detects it very quickly.")
print("When there is no effect, SPRT also concludes faster than running to the fixed horizon.")
print("This is the key advantage: SPRT saves time in BOTH directions.")

---

## 8. Batch SPRT: Daily Analysis at QuickCart

In practice, you rarely process one observation at a time. QuickCart accumulates data throughout the day and runs analysis each morning. **Batch SPRT** processes data in chunks (e.g., daily batches) while preserving the same guarantees.

### Why Batch Processing Works

The SPRT accumulates a sum of log-likelihood ratios. Whether you add them one-at-a-time or in batches, the cumulative sum at any observation count is identical. The only difference: you might overshoot the boundary slightly within a batch (you detect the crossing at the end of the batch rather than at the exact observation).

This makes batch SPRT slightly more conservative (the overshoot means actual error rates are slightly below the nominal levels).

In [ ]:
class BatchSequentialTester(SequentialTester):
    """
    SPRT with batch processing -- checks boundaries only at batch boundaries.
    
    Simulates the practical scenario where you analyze data
    once per day rather than after every single observation.
    """
    
    def __init__(self, effect_size, alpha=0.05, beta=0.2, sigma=1.0):
        super().__init__(effect_size, alpha, beta, sigma)
        self.batch_history = []  # cumulative sum at end of each batch
        self.batch_sizes = []
    
    def add_daily_batch(self, x_batch):
        """
        Process an entire day's worth of data at once.
        Only checks boundaries AFTER the full batch is processed.
        """
        if self.decision is not None:
            return self.decision
        
        # Accumulate all observations in the batch
        for x in x_batch:
            llr = self._log_likelihood_ratio(x)
            self.cumulative_sum += llr
            self.n_observations += 1
            self.history.append(self.cumulative_sum)
        
        # Record batch-level state
        self.batch_history.append(self.cumulative_sum)
        self.batch_sizes.append(len(x_batch))
        
        # Check boundaries only at batch boundary
        if self.cumulative_sum >= self.upper_bound:
            self.decision = 'reject_H0'
        elif self.cumulative_sum <= self.lower_bound:
            self.decision = 'reject_H1'
        
        return self.decision

In [ ]:
# QuickCart scenario: 200 users per day, testing a real effect
np.random.seed(42)
effect_size = 0.3
users_per_day = 200
max_days = 30

tester = BatchSequentialTester(effect_size=effect_size, sigma=1.0)

print("QuickCart Daily SPRT Monitoring")
print("=" * 55)
print(f"Effect size: {effect_size}, Users/day: {users_per_day}")
print(f"Upper boundary (B): {tester.upper_bound:.3f}")
print(f"Lower boundary (A): {tester.lower_bound:.3f}")
print(f"\n{'Day':>4} {'N total':>8} {'Cum LLR':>10} {'Decision':>12}")
print("-" * 40)

for day in range(1, max_days + 1):
    daily_data = np.random.normal(effect_size, 1.0, users_per_day)  # H1 true
    decision = tester.add_daily_batch(daily_data)
    
    status = 'Continue' if decision is None else decision
    print(f"{day:>4} {tester.n_observations:>8} {tester.cumulative_sum:>10.3f} {status:>12}")
    
    if decision is not None:
        n_fixed = fixed_horizon_sample_size(effect_size, 1.0)
        print(f"\nDecision reached on day {day}: {decision}")
        print(f"Total observations used: {tester.n_observations}")
        print(f"Fixed-horizon would need: {n_fixed} observations")
        print(f"Savings: {(1 - tester.n_observations/n_fixed)*100:.0f}%")
        break

In [ ]:
# Visualize the daily monitoring path
fig, ax = plt.subplots(figsize=(12, 5))

# Plot observation-level path
ax.plot(range(1, len(tester.history) + 1), tester.history,
        'b-', alpha=0.3, linewidth=0.5, label='Observation-level')

# Plot batch-level checkpoints
batch_x = np.cumsum(tester.batch_sizes)
ax.plot(batch_x, tester.batch_history, 'ko-', markersize=6, linewidth=2,
        label='Daily batch checkpoints')

# Boundaries
ax.axhline(tester.upper_bound, color='green', linestyle='--', linewidth=2,
           label=f'Upper B = {tester.upper_bound:.2f} (reject H0)')
ax.axhline(tester.lower_bound, color='red', linestyle='--', linewidth=2,
           label=f'Lower A = {tester.lower_bound:.2f} (reject H1)')
ax.axhline(0, color='gray', linestyle='-', alpha=0.3)

# Shade the continue zone
ax.fill_between([0, len(tester.history)], tester.lower_bound, tester.upper_bound,
                alpha=0.05, color='yellow', label='Continue zone')

ax.set_xlabel('Observation number')
ax.set_ylabel('Cumulative log-likelihood ratio')
ax.set_title('Batch SPRT: Daily Monitoring at QuickCart (effect=0.3)')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Batch Size Effect on Stopping Time

How does the batch size affect efficiency? Larger batches mean fewer decision points, but the difference is typically small.

In [ ]:
np.random.seed(42)
batch_sizes = [1, 10, 50, 100, 200, 500]
n_simulations = 3000
effect_size = 0.3

results_by_batch = {}

for bs in batch_sizes:
    samples_needed = []
    for _ in range(n_simulations):
        tester = BatchSequentialTester(effect_size=effect_size, sigma=1.0)
        data = np.random.normal(effect_size, 1.0, 5000)
        
        for i in range(0, len(data), bs):
            batch = data[i:i + bs]
            decision = tester.add_daily_batch(batch)
            if decision is not None:
                break
        samples_needed.append(tester.n_observations)
    
    results_by_batch[bs] = samples_needed

print(f"Effect of batch size on stopping time (H1 true, effect={effect_size})")
print(f"{'Batch Size':>12} {'Mean Samples':>14} {'Median Samples':>16} {'Overhead vs bs=1':>18}")
print("-" * 65)

baseline_mean = np.mean(results_by_batch[1])
for bs in batch_sizes:
    mean_s = np.mean(results_by_batch[bs])
    median_s = np.median(results_by_batch[bs])
    overhead = (mean_s - baseline_mean) / baseline_mean * 100
    print(f"{bs:>12} {mean_s:>14.0f} {median_s:>16.0f} {overhead:>17.1f}%")

print(f"\nDaily batches (size ~200) add minimal overhead compared to observation-level.")
print(f"This justifies the practical choice of checking once per day.")

---

## 9. Operating Characteristics: Expected Sample Size

### Average Sample Number (ASN)

The ASN tells us how many observations SPRT needs on average as a function of the **true** effect. Wald derived approximate formulas:

Under $H_0$ (true effect = 0):
$$
E[N | H_0] \approx \frac{(1-\alpha) \cdot A + \alpha \cdot B}{-\delta^2 / (2\sigma^2)}
$$

Under $H_1$ (true effect = $\delta$):
$$
E[N | H_1] \approx \frac{\beta \cdot A + (1-\beta) \cdot B}{\delta^2 / (2\sigma^2)}
$$

The ASN is highest near the midpoint between H0 and H1, where evidence is most ambiguous.

In [ ]:
# Compute ASN for different true effect sizes
effect_size = 0.3
alpha, beta = 0.05, 0.2
sigma = 1.0

true_effects = np.linspace(-0.1, 0.6, 15)
empirical_asn = []
n_fixed = fixed_horizon_sample_size(effect_size, sigma, alpha, beta)

np.random.seed(42)
for true_eff in true_effects:
    samples = []
    for _ in range(3000):
        tester = SequentialTester(effect_size=effect_size, alpha=alpha, beta=beta, sigma=sigma)
        data = np.random.normal(true_eff, sigma, n_fixed * 3)
        _, n_obs = tester.run_test(data)
        samples.append(n_obs)
    empirical_asn.append(np.mean(samples))

plt.plot(true_effects, empirical_asn, 'bo-', linewidth=2, markersize=8, label='SPRT (empirical ASN)')
plt.axhline(n_fixed, color='red', linestyle='--', linewidth=2,
            label=f'Fixed-horizon (n={n_fixed})')
plt.axvline(0, color='gray', linestyle=':', alpha=0.5)
plt.axvline(effect_size, color='gray', linestyle=':', alpha=0.5)

# Annotate
plt.annotate('H0', xy=(0, n_fixed*0.1), fontsize=11, ha='center', color='gray')
plt.annotate('H1', xy=(effect_size, n_fixed*0.1), fontsize=11, ha='center', color='gray')

plt.xlabel('True effect size')
plt.ylabel('Average sample number (ASN)')
plt.title('SPRT Operating Characteristic: Expected Sample Size vs True Effect')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Key insight: SPRT uses fewer samples than fixed-horizon everywhere")
print(f"EXCEPT near the midpoint (effect ~ {effect_size/2:.2f}), where evidence is most ambiguous.")
print(f"\nAt the design points:")
print(f"  ASN under H0 (effect=0):     {empirical_asn[np.argmin(np.abs(true_effects))]:.0f}")
print(f"  ASN under H1 (effect={effect_size}): {empirical_asn[np.argmin(np.abs(true_effects - effect_size))]:.0f}")
print(f"  Fixed-horizon always uses:   {n_fixed}")

---

## 10. Practical Considerations

### Choosing the Effect Size Parameter

The `effect_size` in SPRT is not "the true effect" -- it's the **smallest effect you want to be able to detect**. Choosing this well is critical:

| Choice | Consequence |
|--------|-------------|
| Too small | Boundaries are far apart; test takes very long to reach a decision |
| Too large | Boundaries are close; fast decisions but may miss smaller real effects |
| Just right | Matches your MDE (minimum detectable effect) from power analysis |

**Rule of thumb**: Use the same MDE you would use for fixed-horizon sample size calculation.

In [ ]:
# How does the choice of effect_size parameter affect SPRT behavior?
np.random.seed(42)
true_effect = 0.3  # the actual effect in the data
assumed_effects = [0.1, 0.2, 0.3, 0.5, 0.8]
n_simulations = 3000

print(f"True effect = {true_effect}, sigma = 1.0")
print(f"How does the assumed effect_size parameter affect SPRT behavior?")
print(f"\n{'Assumed delta':>14} {'Median N':>10} {'P(reject H0)':>14} {'P(reject H1)':>14}")
print("-" * 55)

for assumed in assumed_effects:
    samples = []
    decisions = []
    for _ in range(n_simulations):
        tester = SequentialTester(effect_size=assumed, sigma=1.0)
        data = np.random.normal(true_effect, 1.0, 10000)
        decision, n_obs = tester.run_test(data)
        samples.append(n_obs)
        decisions.append(decision)
    
    p_reject_h0 = sum(1 for d in decisions if d == 'reject_H0') / n_simulations
    p_reject_h1 = sum(1 for d in decisions if d == 'reject_H1') / n_simulations
    print(f"{assumed:>14.1f} {np.median(samples):>10.0f} {p_reject_h0:>14.3f} {p_reject_h1:>14.3f}")

print(f"\nWhen assumed > true: test 'expects' a bigger effect than exists,")
print(f"  so it often concludes 'no effect' (reject H1) -- a Type II error.")
print(f"When assumed < true: test detects the effect but takes longer (wider boundaries).")
print(f"When assumed = true: optimal -- fastest correct decisions.")

### When to Use Sequential vs Fixed-Horizon

| Use Sequential (SPRT) | Use Fixed-Horizon |
|----------------------|-------------------|
| Speed matters: you want to ship winners faster | You only plan to check once at the end |
| You can commit to the effect_size upfront | The effect size is highly uncertain |
| Daily monitoring is operationally needed | Simplicity is more important than speed |
| You also want to stop LOSING experiments early | Regulatory requirements demand fixed design |
| High traffic: many users per day | Low traffic: days between individual observations |

:::{admonition} Modern alternatives to Wald's SPRT
:class: tip
Wald's SPRT requires specifying a fixed alternative $H_1: \mu = \delta$. Modern sequential testing methods relax this:

- **mSPRT (mixture SPRT)**: Averages over a range of alternatives; more robust to misspecification
- **Always-valid confidence sequences**: Provide valid CIs at any stopping time
- **Group sequential methods** (O'Brien-Fleming, Pocock): Allow a fixed number of planned interim looks

Wald's SPRT is the foundation that all these build upon. Understanding it gives you the intuition for the more flexible modern methods.
:::

---

## 11. End-to-End QuickCart Example

### Scenario

QuickCart is A/B testing a new "express checkout" flow. Historical data shows average order value of $85 with standard deviation $30. The team wants to detect a $2 lift.

- alpha = 0.05 (5% false positive rate)
- beta = 0.20 (80% power)
- ~500 users enter the test per day (250 per group)

How many days does it take with SPRT vs fixed-horizon?

In [ ]:
# QuickCart experiment parameters
baseline_aov = 85  # average order value ($)
expected_lift = 2   # expected improvement ($)
sigma_aov = 30      # standard deviation of order values
users_per_day = 500
alpha, beta = 0.05, 0.2

# Fixed-horizon sample size (per group, for a two-sample test)
z_a = stats.norm.ppf(1 - alpha/2)
z_b = stats.norm.ppf(1 - beta)
n_fixed_per_group = int(np.ceil(2 * ((z_a + z_b) * sigma_aov / expected_lift)**2))
days_fixed = np.ceil(n_fixed_per_group / (users_per_day / 2))

print("QuickCart Express Checkout Experiment Plan")
print("=" * 55)
print(f"  Expected lift: ${expected_lift} (from ${baseline_aov} baseline)")
print(f"  Metric std dev: ${sigma_aov}")
print(f"  Users per day: {users_per_day} (250 per group)")
print(f"\nFixed-horizon plan:")
print(f"  Sample size per group: {n_fixed_per_group:,}")
print(f"  Days needed: {days_fixed:.0f}")

# Simulate SPRT for the difference metric
# Model: each observation is the difference (treatment AOV - baseline)
np.random.seed(42)
n_simulations = 3000

# Case 1: Effect is real (lift = $2)
days_to_decision_h1 = []
for _ in range(n_simulations):
    tester = BatchSequentialTester(
        effect_size=expected_lift, alpha=alpha, beta=beta, sigma=sigma_aov
    )
    for day in range(1, 300):
        # Observe treatment users' deviation from baseline
        daily_obs = np.random.normal(expected_lift, sigma_aov, users_per_day // 2)
        decision = tester.add_daily_batch(daily_obs)
        if decision is not None:
            days_to_decision_h1.append(day)
            break
    else:
        days_to_decision_h1.append(300)

# Case 2: No effect (lift = $0)
days_to_decision_h0 = []
for _ in range(n_simulations):
    tester = BatchSequentialTester(
        effect_size=expected_lift, alpha=alpha, beta=beta, sigma=sigma_aov
    )
    for day in range(1, 300):
        daily_obs = np.random.normal(0, sigma_aov, users_per_day // 2)
        decision = tester.add_daily_batch(daily_obs)
        if decision is not None:
            days_to_decision_h0.append(day)
            break
    else:
        days_to_decision_h0.append(300)

print(f"\nSPRT Results (daily batches of {users_per_day//2} per group):")
print(f"\n  When effect IS real (${expected_lift} lift):")
print(f"    Median days to decision: {np.median(days_to_decision_h1):.0f}")
print(f"    Mean days to decision:   {np.mean(days_to_decision_h1):.0f}")
print(f"\n  When NO effect (null is true):")
print(f"    Median days to decision: {np.median(days_to_decision_h0):.0f}")
print(f"    Mean days to decision:   {np.mean(days_to_decision_h0):.0f}")
print(f"\n  Fixed-horizon comparison: {days_fixed:.0f} days (regardless)")
print(f"  SPRT saves ~{days_fixed - np.median(days_to_decision_h1):.0f} days when effect is real")
print(f"  SPRT saves ~{days_fixed - np.median(days_to_decision_h0):.0f} days when no effect")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(days_to_decision_h1, bins=30, density=True, alpha=0.7,
             color='darkorange', edgecolor='black', linewidth=0.5)
axes[0].axvline(days_fixed, color='red', linestyle='--', linewidth=2,
               label=f'Fixed-horizon: {days_fixed:.0f} days')
axes[0].axvline(np.median(days_to_decision_h1), color='black', linewidth=2,
               label=f'SPRT median: {np.median(days_to_decision_h1):.0f} days')
axes[0].set_xlabel('Days to decision')
axes[0].set_ylabel('Density')
axes[0].set_title('Days to Decision When Effect IS Real ($2 lift)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(days_to_decision_h0, bins=30, density=True, alpha=0.7,
             color='steelblue', edgecolor='black', linewidth=0.5)
axes[1].axvline(days_fixed, color='red', linestyle='--', linewidth=2,
               label=f'Fixed-horizon: {days_fixed:.0f} days')
axes[1].axvline(np.median(days_to_decision_h0), color='black', linewidth=2,
               label=f'SPRT median: {np.median(days_to_decision_h0):.0f} days')
axes[1].set_xlabel('Days to decision')
axes[1].set_ylabel('Density')
axes[1].set_title('Days to Decision When No Effect')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('QuickCart Express Checkout: SPRT Stopping Time Distributions',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| Peeking problem | Checking p-values repeatedly inflates FPR from 5% to 20-30%+ |
| Sequential testing | Designed for continuous monitoring; controls errors at ALL stopping times |
| SPRT core idea | Accumulate log-likelihood ratio; stop when it crosses a boundary |
| Upper boundary (B) | $\log\frac{1-\beta}{\alpha}$; crossing means "effect is real" (reject H0) |
| Lower boundary (A) | $\log\frac{\beta}{1-\alpha}$; crossing means "no meaningful effect" (reject H1) |
| Log-likelihood ratio | For normal data: $\frac{\delta}{\sigma^2}(x_i - \delta/2)$ per observation |
| Sample savings | SPRT typically needs 30-60% fewer observations than fixed-horizon |
| Batch SPRT | Check boundaries once per day; minimal overhead vs observation-level |
| ASN curve | Highest when true effect is near $\delta/2$ (most ambiguous region) |
| Effect size parameter | Use your MDE; too small = slow, too large = misses smaller effects |
| When to use | High traffic, daily monitoring needed, want to ship faster |

---

## Course Conclusion: The 8-Week Journey

Congratulations on completing the Advanced A/B Testing course! Here is a recap of what you have built over 8 weeks:

| Week | Topic | What You Learned |
|------|-------|------------------|
| 1 | Statistical Foundations | t-tests, bootstrap, Type I/II errors, AAB validation |
| 2 | Real-World Analysis | Messy metrics, outlier handling, practical test pipelines |
| 3 | Sample Size and MDE | Power analysis, minimum detectable effect, experiment planning |
| 4 | Stratification | Stratified sampling, post-stratification for variance reduction |
| 5 | CUPED | Pre-experiment covariates for variance reduction, ML-enhanced CUPED |
| 6 | Ratio Metrics | Linearization, delta method, revenue-per-session analysis |
| 7 | Multiple Testing | FWER, FDR, Bonferroni, Benjamini-Hochberg, concurrent experiments |
| 8 | Sequential Testing | SPRT, continuous monitoring, early stopping, batch analysis |

### The Full Stack of Experimentation

Together, these techniques form a complete experimentation toolkit:

1. **Before the experiment**: Power analysis (Week 3) determines how long to run. Choose your metric carefully -- ratio metrics need special handling (Week 6).

2. **During the experiment**: CUPED (Week 5) and stratification (Week 4) reduce variance so you can detect effects faster. Sequential testing (Week 8) lets you monitor safely and stop early. Guard against multiple testing issues (Week 7) when running concurrent experiments.

3. **After the experiment**: Apply the right statistical test (Week 1), handle real-world messiness (Week 2), and make sound decisions with calibrated uncertainty.

### Key Principles to Carry Forward

- **Always validate**: Run A/A tests before trusting any methodology
- **Reduce variance, not bias**: CUPED, stratification, and proper metrics improve sensitivity without compromising validity
- **Design for the analysis you want**: Sequential monitoring requires pre-commitment to the framework
- **Communicate clearly**: Statistical rigor means nothing if stakeholders do not understand or trust the results
- **Iterate on the platform**: Each technique you add makes every future experiment more efficient

The experimentation platform at QuickCart -- and wherever you work -- is never "done." Each week's technique is a tool you can deploy to make experiments faster, cheaper, and more reliable.